# Station charges — 02 calibration

Turns the registered sources into one charge per stop, in
`data/station_charges.csv`, which `step10_export_seed_stops.py` joins onto the
catalog. Notebook is truth; the CSV is generated and gitignored.

Each source gets **its own reader section** below, because the formats differ:
a station price list is a PDF table, a network statement annex may be an XLSX,
and some figures can only be typed in by hand from a scan. Every reader, however
it works, appends rows in one shape:

```python
{"source_ref": "<source_id from 01>", "station": "<name as printed>",
 "country_code": "DE", "charge_eur": 9.80, "basis": "per_call", "note": "..."}
```

The resolve step then does the two jobs common to all of them: match the
printed station name to a catalog `stop_id`, and pick one value per stop when
sources overlap.

## Adding a source

1. Register the document in `01_source_extraction.ipynb` and put the file in
   `sources/`.
2. Copy the nearest reader section below and adapt it.
3. Re-run this notebook top to bottom, then
   `python ../step10_export_seed_stops.py`.

Only real sourced values. If a figure cannot be read reliably, leave the stop
out — an absent stop resolves through the global default (11.28 EUR), which is
honest, whereas a guessed number silently overrides it.

In [1]:
import csv
import re
import unicodedata
from pathlib import Path


def _resolve_dirs() -> tuple[Path, Path]:
    here = Path.cwd()
    for base in (here, here / "backend/models/infrastructure/stops/charges"):
        if (base / "data").exists() or base.name == "charges":
            data = base / "data"
            data.mkdir(exist_ok=True)
            return data, base / "sources"
    raise RuntimeError(f"cannot locate the charges directories from {here}")


DATA_DIR, SOURCES_DIR = _resolve_dirs()
CATALOG_PATH = DATA_DIR.parent.parent / "data" / "stop_seed_catalog.csv"
OUTPUT_PATH = DATA_DIR / "station_charges.csv"

OUTPUT_COLUMNS = [
    "stop_id",
    "stop_name",
    "country_code",
    "stop_charge_eur",
    "basis",
    "source_ref",
    "note",
]

# Every reader appends here.
readings: list[dict] = []

register = {
    row["source_id"]: row
    for row in csv.DictReader(open(DATA_DIR / "sources_register.csv", encoding="utf-8-sig"))
}
print(f"{len(register)} registered sources")

2 registered sources


## Resolving a printed name to a catalog stop

Tariff documents print station names, the catalog is keyed by OSM id, and the
two rarely agree on spelling. `resolve()` matches within a country using the
same normalisation step 5 uses — transliteration, parenthetical qualifiers
dropped, station abbreviations expanded — and refuses ambiguous hits rather
than guessing, so an unresolved name is reported instead of silently charging
the wrong station.

In [2]:
ABBREVIATIONS = {
    "hbf": "hauptbahnhof",
    "hb": "hauptbahnhof",
    "bf": "bahnhof",
    "bhf": "bahnhof",
    "centraal": "central",
    "centrale": "central",
    "gl": "glowny",
    "st": "sankt",
}


def normalize(name: str) -> str:
    text = unicodedata.normalize("NFKD", name or "")
    text = text.encode("ascii", "ignore").decode().lower()
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text).strip()
    return " ".join(ABBREVIATIONS.get(word, word) for word in text.split())


catalog: list[dict] = []
if CATALOG_PATH.is_file():
    with open(CATALOG_PATH, encoding="utf-8-sig", newline="") as fh:
        catalog = list(csv.DictReader(fh))
else:
    print(
        f"  WARNING: {CATALOG_PATH.name} not found — run "
        "step10_export_seed_stops.py once first, then re-run this notebook so "
        "names can be resolved to stop ids."
    )

by_country: dict[str, dict[str, list[dict]]] = {}
for row in catalog:
    bucket = by_country.setdefault(row["country_code"], {})
    bucket.setdefault(normalize(row["stop_name"]), []).append(row)

unresolved: list[tuple[str, str, str]] = []


def resolve(station: str, country_code: str, source_ref: str):
    """Catalog row for a printed station name, or None (and reported)."""
    candidates = by_country.get(country_code, {}).get(normalize(station), [])
    if len(candidates) == 1:
        return candidates[0]
    reason = "no catalog stop of that name" if not candidates else (
        f"{len(candidates)} catalog stops share the name — disambiguate by id"
    )
    unresolved.append((source_ref, f"{station} ({country_code})", reason))
    return None

## Reader — manual transcription

The simplest path, and the right one whenever a document cannot be parsed
reliably: type the figures in, citing the page. Also where the thirteen
placeholder charges inherited from the retired curated catalog live — each one
is a value to **replace**, not to trust.

In [3]:
# --- ILLUSTRATIVE-CURATED: placeholders from the retired curated catalog ---
# NOT published tariffs. Delete a row here as soon as the station has a sourced
# figure from a real document.
_ILLUSTRATIVE = [
    ("Berlin Hauptbahnhof", "DE", 9.80),
    ("Dresden Hbf", "DE", 6.50),
    ("Hamburg Hauptbahnhof", "DE", 32.47),
    ("Köln Hauptbahnhof", "DE", 47.34),
    ("Wien Hauptbahnhof", "AT", 11.00),
    ("Graz Hauptbahnhof", "AT", 50.50),
    ("Zürich Hauptbahnhof", "CH", 14.50),
    ("Basel SBB", "CH", 73.86),
    ("Paris Gare de l'Est", "FR", 13.20),
    ("Bruxelles-Midi - Brussel-Zuid", "BE", 10.40),
    ("Liège-Guillemins", "BE", 126.67),
    ("Københavns Hovedbanegård", "DK", 9.00),
    ("Odense", "DK", 93.38),
]

readings += [
    {
        "source_ref": "ILLUSTRATIVE-CURATED",
        "station": station,
        "country_code": country,
        "charge_eur": charge,
        "basis": "per_call",
        "note": "illustrative placeholder — replace with a sourced figure",
    }
    for station, country, charge in _ILLUSTRATIVE
]
print(f"manual transcription: {len(readings)} readings")

manual transcription: 13 readings


## Reader — XLSX annex

For network statement annexes and operator models published as spreadsheets.
Reads with `openpyxl` (already a backend dependency, used by the ONTD loaders).
Adapt the column names to the sheet; keep the try/except so a missing file
skips the source instead of breaking the run.

In [4]:
def read_xlsx_charges(
    filename: str,
    source_ref: str,
    country_code: str,
    *,
    sheet: str | None = None,
    station_column: str = "Station",
    charge_column: str = "Entgelt",
    basis: str = "per_call",
) -> list[dict]:
    """Station/charge pairs from a spreadsheet annex.

    Header row is found by looking for both column names, so a sheet with a
    title block above the table still reads. Returns [] with a message when
    the file is absent — an unavailable source must not stop the others.
    """
    path = SOURCES_DIR / filename
    if not path.is_file():
        print(f"  {filename} not in sources/ — skipped")
        return []

    from openpyxl import load_workbook

    workbook = load_workbook(path, data_only=True, read_only=True)
    worksheet = workbook[sheet] if sheet else workbook.active

    header, columns = None, {}
    rows_out = []
    for row in worksheet.iter_rows(values_only=True):
        values = ["" if v is None else str(v).strip() for v in row]
        if header is None:
            if station_column in values and charge_column in values:
                header = values
                columns = {name: i for i, name in enumerate(values)}
            continue

        station = values[columns[station_column]]
        raw = values[columns[charge_column]].replace(",", ".")
        if not station or not raw:
            continue
        try:
            charge = float(re.sub(r"[^0-9.]", "", raw))
        except ValueError:
            continue
        rows_out.append(
            {
                "source_ref": source_ref,
                "station": station,
                "country_code": country_code,
                "charge_eur": charge,
                "basis": basis,
                "note": f"{filename}, sheet {worksheet.title}",
            }
        )

    if header is None:
        print(f"  {filename}: no header row with {station_column!r}/{charge_column!r}")
    print(f"  {filename}: {len(rows_out)} readings")
    return rows_out


# Josh: uncomment and adapt once a spreadsheet source is in sources/.
# readings += read_xlsx_charges(
#     "at_oebb_snnb_2026_annex.xlsx", "AT-SNNB-2026", "AT",
#     station_column="Bahnhof", charge_column="Stationsentgelt",
# )

## Reader — PDF table

Many station price lists publish a machine-readable table. `pdfplumber` is not
a backend dependency, so this reader installs nothing by default: run the
notebook with `uv run --with pdfplumber ...` when a PDF source needs it, and it
becomes available.

If a document turns out to be a scan, or the table extraction is unreliable,
**do not fight it** — transcribe the figures into the manual section above with
the page cited. A transcribed number with a page reference is worth more than a
parsed one nobody can check.

In [5]:
def read_pdf_charges(
    filename: str,
    source_ref: str,
    country_code: str,
    *,
    pages: str | None = None,
    station_column: int = 0,
    charge_column: int = -1,
    basis: str = "per_call",
) -> list[dict]:
    """Station/charge pairs from a PDF table, by column position.

    Column indices rather than names: price list tables are frequently
    unheadered or repeat the header on every page. Check a sample of the output
    against the document before trusting a run.
    """
    path = SOURCES_DIR / filename
    if not path.is_file():
        print(f"  {filename} not in sources/ — skipped")
        return []
    try:
        import pdfplumber
    except ImportError:
        print(
            f"  {filename}: pdfplumber not installed — re-run the notebook with "
            "`uv run --project ../../../../ --with pdfplumber jupyter lab`, or "
            "transcribe the figures into the manual section instead"
        )
        return []

    wanted = None if pages is None else {int(p) for p in pages.split(",")}
    rows_out = []
    with pdfplumber.open(path) as pdf:
        for number, page in enumerate(pdf.pages, start=1):
            if wanted is not None and number not in wanted:
                continue
            for table in page.extract_tables() or []:
                for row in table:
                    cells = [(c or "").strip() for c in row]
                    if len(cells) < 2:
                        continue
                    station = cells[station_column]
                    raw = cells[charge_column].replace(",", ".")
                    if not station or not re.search(r"\d", raw):
                        continue
                    try:
                        charge = float(re.sub(r"[^0-9.]", "", raw))
                    except ValueError:
                        continue
                    rows_out.append(
                        {
                            "source_ref": source_ref,
                            "station": station,
                            "country_code": country_code,
                            "charge_eur": charge,
                            "basis": basis,
                            "note": f"{filename}, page {number}",
                        }
                    )
    print(f"  {filename}: {len(rows_out)} readings")
    return rows_out


# Josh: uncomment and adapt once a PDF source is in sources/.
# readings += read_pdf_charges(
#     "de_db_stationspreisliste_2026.pdf", "DE-DB-SPL-2026", "DE", pages="12,13",
# )

## Resolve

Every reading is matched to a catalog stop and, where sources overlap, one
value wins. The rule is deliberately blunt: **a sourced figure always beats an
illustrative one**, and among equals the later `price_basis_year` wins. Any
other collision is reported rather than silently resolved — two real tariffs
disagreeing on one station is a question for a human, not a tie-break.

In [6]:
def priority(reading: dict) -> tuple[int, int]:
    source = register.get(reading["source_ref"], {})
    sourced = 0 if source.get("kind") == "manual_transcription" and (
        reading["source_ref"] == "ILLUSTRATIVE-CURATED"
    ) else 1
    try:
        year = int(source.get("price_basis_year") or 0)
    except ValueError:
        year = 0
    return sourced, year


unknown_refs = sorted({r["source_ref"] for r in readings} - set(register))
if unknown_refs:
    raise KeyError(
        f"readings cite sources absent from the register: {unknown_refs} — "
        "add them in 01_source_extraction.ipynb"
    )

best: dict[str, dict] = {}
collisions: list[str] = []
for reading in readings:
    stop = resolve(reading["station"], reading["country_code"], reading["source_ref"])
    if stop is None:
        continue
    stop_id = stop["stop_id"]
    current = best.get(stop_id)
    if current is None:
        best[stop_id] = {"stop": stop, "reading": reading}
        continue
    new, old = priority(reading), priority(current["reading"])
    if new > old:
        best[stop_id] = {"stop": stop, "reading": reading}
    elif new == old and reading["charge_eur"] != current["reading"]["charge_eur"]:
        collisions.append(
            f"{stop['stop_name']}: {current['reading']['source_ref']} says "
            f"{current['reading']['charge_eur']}, {reading['source_ref']} says "
            f"{reading['charge_eur']}"
        )

print(f"{len(readings)} readings -> {len(best)} stops")
if unresolved:
    print(f"\n{len(unresolved)} unresolved station names:")
    for source_ref, station, reason in unresolved:
        print(f"  [{source_ref}] {station}: {reason}")
if collisions:
    print(f"\n{len(collisions)} equal-priority disagreements — resolve by hand:")
    for line in collisions:
        print(f"  {line}")

13 readings -> 13 stops


## Write

In [7]:
rows = []
for stop_id, entry in best.items():
    stop, reading = entry["stop"], entry["reading"]
    rows.append(
        {
            "stop_id": stop_id,
            "stop_name": stop["stop_name"],
            "country_code": stop["country_code"],
            "stop_charge_eur": f"{reading['charge_eur']:.2f}",
            "basis": reading["basis"],
            "source_ref": reading["source_ref"],
            "note": reading["note"],
        }
    )
rows.sort(key=lambda r: (r["country_code"], r["stop_name"]))

with open(OUTPUT_PATH, "w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=OUTPUT_COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

illustrative = sum(1 for r in rows if r["source_ref"] == "ILLUSTRATIVE-CURATED")
print(f"wrote {len(rows)} rows to {OUTPUT_PATH.name}")
print(f"  {illustrative} still illustrative, {len(rows) - illustrative} sourced")
print(f"  {len(catalog) - len(rows)} catalog stops have no charge (global default)")

wrote 13 rows to station_charges.csv
  13 still illustrative, 0 sourced
  967 catalog stops have no charge (global default)
